In [1]:
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, WeightedRandomSampler
import wandb
from sklearn.metrics import precision_score
from accelerate import Accelerator
from accelerate import DistributedType, DistributedDataParallelKwargs
from peft import get_peft_model, LoraConfig, TaskType
import os
from utils.utils import seed_everything
from transformers import LongformerTokenizer
from datasets import EHR_Longformer_Dataset
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import torch.nn.functional as F
from models.longformernormal import LongformerPretrainNormal, LongformerFinetune
from torch.optim.lr_scheduler import LinearLR, SequentialLR, ExponentialLR, LambdaLR, CosineAnnealingWarmRestarts, ReduceLROnPlateau
from finetune_train import train
import logging
import sys
from collections import Counter
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import Sampler
from torch.optim.lr_scheduler import _LRScheduler
import math
import torch.nn.init as init
import warnings
from torch.utils.data import WeightedRandomSampler, DataLoader
import numpy as np
from collections import Counter
import random
from torch.utils.data import Dataset, DataLoader, Sampler
import torch.distributed as dist
from utils.sampler import RandomOversamplingDistributedSampler

In [2]:
parser = argparse.ArgumentParser()
    
# Required parameters
parser.add_argument("--exp_name", type=str, default="mortality30")
parser.add_argument("--save_path", type=str, default="./results")
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--checkpoint_dir", type=str, default="./checkpoints")

# Model parameters
parser.add_argument("--mode", type=str, default="mortality30")
parser.add_argument("--vocab_size", type=int, default=50265)
parser.add_argument("--itemid_size", type=int, default=4016)
parser.add_argument("--unit_size", type=int, default=60)
parser.add_argument("--gender_size", type=int, default=2)
parser.add_argument("--continuous_size", type=int, default=3)
parser.add_argument("--task_size", type=int, default=20)
parser.add_argument("--max_position_embeddings", type=int, default=5000)
parser.add_argument("--max_age", type=int, default=100)
parser.add_argument("--batch_size", type=int, default=2)
parser.add_argument("--resume", type=bool, default=False)
parser.add_argument("--pin_memory", type=bool, default=True)
parser.add_argument("--nodes", type=int, default=1)
parser.add_argument("--gpus", type=int, default=2)
parser.add_argument("--start_epoch", type=int, default=0)
parser.add_argument("--epochs", type=int, default=200)
parser.add_argument("--log_every_n_steps", type=int, default=100)
parser.add_argument("--acc", type=int, default=8)
parser.add_argument("--resume_checkpoint", type=str, default=None)
parser.add_argument("--num_workers", type=int, default=0)
parser.add_argument("--embedding_size", type=int, default=768)
parser.add_argument("--num_hidden_layers", type=int, default=1)
parser.add_argument("--num_attention_heads", type=int, default=1)
parser.add_argument("--intermediate_size", type=int, default=3072)
parser.add_argument("--learning_rate", type=float, default=1e-4)
parser.add_argument("--dropout_prob", type=float, default=0.1)
parser.add_argument("--lora_dropout", type=float, default=0.1)
parser.add_argument("--classifier_dropout", type=float, default=0.1)
parser.add_argument("--device", type=str, default="cuda")
parser.add_argument("--gpu_mixed_precision", type=bool, default=True)
parser.add_argument("--patience", type=int, default=10)
parser.add_argument("--num_labels", type=int, default=1)
parser.add_argument("--use_lora", action='store_true', default=False)
parser.add_argument("--gamma", type=float, default=2.0)
parser.add_argument("--beta", type=float, default=0.99)
parser.add_argument('--lora_weight', type=int, default=2)
parser.add_argument('--classifier_weight', type=int, default=5)
parser.add_argument("--loss", type=str, default="cross_entropy")
parser.add_argument("--pretrain", action='store_true', default=False)
parser.add_argument("--clip_interval", type=int, default=1)
parser.add_argument("--pretrain_path", type=str, default="best_pretrain_model.pth")
parser.add_argument("--lora_layers", type=list, default=[8,9,10,11])
parser.add_argument("--lora_r", type=int, default=4)
parser.add_argument("--lora_alpha", type=int, default=8)
parser.add_argument("--loss_alpha", type=float, default=0.5)
parser.add_argument("--regression_mode", type=bool, default=False)
parser.add_argument("--similarity_mode", type=bool, default=False)
parser.add_argument("--similarity_factor", type=float, default=0.25)
parser.add_argument("--percent", type=str, default="10percent")
parser.add_argument("--debug", type=bool, default=False)
parser.add_argument("--freeze", type=bool, default=False)
parser.add_argument("--oversampling_ratio", type=float, default=1.0)
parser.add_argument("--mask_mode", type=str, default='mlm')

args = parser.parse_args()
args.attention_window = [512] * args.num_hidden_layers


In [3]:
tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")

itemid2idx = pd.read_pickle("datasets/entire_itemid2idx.pkl")
unit2idx = pd.read_pickle("datasets/unit2idx.pkl")
idx2label = pd.read_pickle("datasets/idx2label.pkl") ##############
vocab_size = len(itemid2idx.keys())
vocab_size

/home/DAHS3/anaconda3/envs/sj/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


4016

In [4]:
import sys
import traceback

try:
    # 오류 발생 가능성이 있는 코드
    train_dataset = EHR_Longformer_Dataset(Path("./datasets"), "train", tokenizer, itemid2idx, unit2idx, vocab_size=vocab_size, use_itemid=True, mode="pretrain", percent=args.percent, mask_mode='span_mlm')
    train_dataset.__getitem__(0)
    raise ValueError("테스트 예외 발생 🚀")
except Exception as e:
    print("⚠️ 예외 발생: 기본 방식으로 출력")
    print(e)  # 기본 예외 메시지 출력

    print("\n📌 상세 오류 정보:")
    exc_type, exc_value, exc_traceback = sys.exc_info()
    traceback.print_exception(exc_type, exc_value, exc_traceback)

⚠️ 예외 발생: 기본 방식으로 출력
테스트 예외 발생 🚀

📌 상세 오류 정보:


Traceback (most recent call last):
  File "/tmp/ipykernel_2245699/2458241271.py", line 8, in <module>
    raise ValueError("테스트 예외 발생 🚀")
ValueError: 테스트 예외 발생 🚀


In [19]:
train_dataset = EHR_Longformer_Dataset(Path("./datasets"), "train", tokenizer, itemid2idx, unit2idx, vocab_size=vocab_size, use_itemid=True, mode="pretrain", percent=args.percent, mask_mode='span_mlm')
# valid_dataset = EHR_Longformer_Dataset(Path("./datasets"), "valid", tokenizer, itemid2idx, unit2idx, vocab_size=vocab_size, use_itemid=True, mode=args.mode, percent=args.percent)
# test_dataset = EHR_Longformer_Dataset(Path("./datasets"), "test", tokenizer, itemid2idx, unit2idx, vocab_size=vocab_size, use_itemid=True, mode=args.mode, percent=args.percent)


In [8]:
test = train_dataset.__getitem__(0)

In [10]:
test[1].sum()

tensor(541)

In [12]:
len(test[-1])

541

In [17]:
541*0.15

81.14999999999999

In [16]:
torch.sum(test[-2] != -100)

tensor(48)

In [14]:
test[-1].sum()

tensor(33)

In [5]:
len(train_dataset.__getitem__(0)[-1])

541

In [7]:
len(train_dataset.__getitem__(0)[-2])

4093

In [17]:
tensor_data =train_dataset.__getitem__(0)[-1] 
num_non_masked = torch.sum(tensor_data != -100)

print(num_non_masked)
num_non_masked = torch.count_nonzero(tensor_data != -100)
print(num_non_masked)

tensor(64)
tensor(64)


In [18]:
filtered_tensor = tensor_data[tensor_data != -100]
print(filtered_tensor)

tensor([1649, 3778, 1116, 1431,  398,  679,  797,  798,  804, 3778,   28,   38,
           4,   38,   28,  801,   42,  802,   30,   28,    4,  152,  158, 1439,
        1741,   27,    4,  800,  808,  810,  838,   26,   27, 1649,  807,  811,
          30,   26,   42, 3088, 3774,   27,   30,   38,    4,   38,   27,  157,
         401,   30,   38,   26, 3773,   27,    4,   38,   26,   26,   38,    4,
           4,   38,  339,  326])


In [8]:
train_dataset.__getitem__(0)[-1]

tensor([-100, -100, -100,  ..., -100, -100, -100])